# Comparaison de modèles avec MLflow

Ce notebook compare plusieurs baselines sur la même séparation des données et avec les mêmes métriques métier.

In [13]:
# ---------- Bibliothèques ----------
from pathlib import Path

import mlflow
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, f1_score, make_scorer, recall_score, roc_auc_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# ---------- Chargement des données ----------
DATA_PATH = Path("../data/processed/application_train_final_encoded.csv")
ID_COLUMN = "SK_ID_CURR"
TARGET_COLUMN = "TARGET"
TEST_SIZE = 0.20
RANDOM_STATE = 42

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Fichier introuvable : {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
required_columns = {ID_COLUMN, TARGET_COLUMN}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Colonnes obligatoires absentes : {sorted(missing_columns)}")

# ---------- Variable cible et séparation entraînement-validation ----------
X = df.drop(columns=[ID_COLUMN, TARGET_COLUMN])
y = df[TARGET_COLUMN]
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)
SCALE_POS_WEIGHT = (y_train == 0).sum() / (y_train == 1).sum()

# ---------- Configuration MLflow ----------
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("credit_scoring")

print(f"Entraînement : {len(X_train):,} lignes")
print(f"Validation : {len(X_valid):,} lignes")

Entraînement : 246,008 lignes
Validation : 61,502 lignes


## Pipeline commun

Tous les modèles utilisent l'imputation médiane. La standardisation est activée uniquement lorsqu'elle est utile. La fonction d'évaluation applique les mêmes métriques et le même suivi MLflow à chaque baseline.

In [14]:
# ---------- Validation croisée ----------
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
RECALL_SCORER = make_scorer(recall_score, pos_label=1)

# ---------- Pipeline de préparation ----------
def create_model_pipeline(classifier, use_scaler=False):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if use_scaler:
        steps.append(("scaler", StandardScaler()))
    steps.append(("classifier", classifier))
    return Pipeline(steps)


# ---------- Entraînement, évaluation et tracking ----------
def run_baseline(
    classifier, run_name, description, model_params, weighting, use_scaler=False
):
    model = create_model_pipeline(classifier, use_scaler)

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("description", description)
        mlflow.set_tag("weighting", weighting)
        mlflow.log_param("model_type", classifier.__class__.__name__)
        mlflow.log_param("test_size", TEST_SIZE)
        mlflow.log_param("cv_folds", CV.get_n_splits())
        mlflow.log_param("imputation", "median")
        mlflow.log_param("scaling", "standard" if use_scaler else "none")
        mlflow.log_params(model_params)

        cv_recall_scores = cross_val_score(
            model, X_train, y_train, scoring=RECALL_SCORER, cv=CV, n_jobs=1
        )
        cv_recall_mean = cv_recall_scores.mean()
        cv_recall_std = cv_recall_scores.std()

        model.fit(X_train, y_train)
        y_pred = model.predict(X_valid)
        y_score = model.predict_proba(X_valid)[:, 1]

        metrics = {
            "roc_auc": roc_auc_score(y_valid, y_score),
            "recall": recall_score(y_valid, y_pred, pos_label=1),
            "f1_score": f1_score(y_valid, y_pred, pos_label=1),
        }
        mlflow.log_metrics(metrics)
        mlflow.log_metrics({
            "cv_recall_mean": cv_recall_mean,
            "cv_recall_std": cv_recall_std,
            "recall_cv_gap": abs(metrics["recall"] - cv_recall_mean),
        })

        tn, fp, fn, tp = confusion_matrix(y_valid, y_pred).ravel()
        mlflow.log_metrics({"fn": int(fn), "fp": int(fp)})

        display(pd.DataFrame(
            [[tn, fp], [fn, tp]],
            index=["Réel : bon (0)", "Réel : mauvais (1)"],
            columns=["Prédit : bon (0)", "Prédit : mauvais (1)"],
        ))
        print(f"TN : {tn} | FP : {fp} | FN : {fn} | TP : {tp}")

    return pd.Series(metrics, name=run_name)

## Régression logistique

In [15]:
# ---------- Configuration de la régression logistique ----------
LOGISTIC_PARAMS = {
    "C": 1.0,
    "solver": "lbfgs",
    "max_iter": 1000,
    "class_weight": None,
    "random_state": RANDOM_STATE,
}

logistic_results = run_baseline(
    LogisticRegression(**LOGISTIC_PARAMS),
    run_name="logistic_regression_baseline",
    description="Baseline de régression logistique",
    model_params=LOGISTIC_PARAMS,
    weighting="unweighted",
    use_scaler=True,
)

# ---------- Variante avec pondération ----------
LOGISTIC_BALANCED_PARAMS = {**LOGISTIC_PARAMS, "class_weight": "balanced"}
logistic_balanced_results = run_baseline(
    LogisticRegression(**LOGISTIC_BALANCED_PARAMS),
    run_name="logistic_regression_balanced",
    description="Régression logistique avec classes équilibrées",
    model_params=LOGISTIC_BALANCED_PARAMS,
    weighting="balanced",
    use_scaler=True,
)

,Prédit : bon (0),Prédit : mauvais (1)
Réel : bon (0),56421,116
Réel : mauvais (1),4824,141


TN : 56421 | FP : 116 | FN : 4824 | TP : 141


,Prédit : bon (0),Prédit : mauvais (1)
Réel : bon (0),40028,16509
Réel : mauvais (1),1524,3441


TN : 40028 | FP : 16509 | FN : 1524 | TP : 3441


## Random Forest

In [16]:
# ---------- Configuration du Random Forest ----------
RANDOM_FOREST_PARAMS = {
    "n_estimators": 100,
    "criterion": "gini",
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "class_weight": None,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

random_forest_results = run_baseline(
    RandomForestClassifier(**RANDOM_FOREST_PARAMS),
    run_name="random_forest_baseline",
    description="Baseline Random Forest",
    model_params=RANDOM_FOREST_PARAMS,
    weighting="unweighted",
)

# ---------- Variante avec pondération ----------
RANDOM_FOREST_BALANCED_PARAMS = {
    **RANDOM_FOREST_PARAMS, "class_weight": "balanced"
}
random_forest_balanced_results = run_baseline(
    RandomForestClassifier(**RANDOM_FOREST_BALANCED_PARAMS),
    run_name="random_forest_balanced",
    description="Random Forest avec classes équilibrées",
    model_params=RANDOM_FOREST_BALANCED_PARAMS,
    weighting="balanced",
)

,Prédit : bon (0),Prédit : mauvais (1)
Réel : bon (0),56537,0
Réel : mauvais (1),4958,7


TN : 56537 | FP : 0 | FN : 4958 | TP : 7


,Prédit : bon (0),Prédit : mauvais (1)
Réel : bon (0),56243,294
Réel : mauvais (1),4729,236


TN : 56243 | FP : 294 | FN : 4729 | TP : 236


## XGBoost

In [17]:
# ---------- Configuration de XGBoost ----------
XGBOOST_PARAMS = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.3,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

xgboost_results = run_baseline(
    XGBClassifier(**XGBOOST_PARAMS),
    run_name="xgboost_baseline",
    description="Baseline XGBoost",
    model_params=XGBOOST_PARAMS,
    weighting="unweighted",
)

# ---------- Variante avec pondération ----------
XGBOOST_BALANCED_PARAMS = {
    **XGBOOST_PARAMS, "scale_pos_weight": SCALE_POS_WEIGHT
}
xgboost_balanced_results = run_baseline(
    XGBClassifier(**XGBOOST_BALANCED_PARAMS),
    run_name="xgboost_balanced",
    description="XGBoost avec pondération de la classe 1",
    model_params=XGBOOST_BALANCED_PARAMS,
    weighting="balanced",
)

,Prédit : bon (0),Prédit : mauvais (1)
Réel : bon (0),56169,368
Réel : mauvais (1),4663,302


TN : 56169 | FP : 368 | FN : 4663 | TP : 302


,Prédit : bon (0),Prédit : mauvais (1)
Réel : bon (0),43551,12986
Réel : mauvais (1),1881,3084


TN : 43551 | FP : 12986 | FN : 1881 | TP : 3084


## LightGBM

In [18]:
# ---------- Configuration de LightGBM ----------
LIGHTGBM_PARAMS = {
    "boosting_type": "gbdt",
    "n_estimators": 100,
    "learning_rate": 0.1,
    "num_leaves": 31,
    "max_depth": -1,
    "objective": "binary",
    "class_weight": None,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
}

lightgbm_results = run_baseline(
    LGBMClassifier(**LIGHTGBM_PARAMS),
    run_name="lightgbm_baseline",
    description="Baseline LightGBM",
    model_params=LIGHTGBM_PARAMS,
    weighting="unweighted",
)

# ---------- Variante avec pondération ----------
LIGHTGBM_BALANCED_PARAMS = {
    **LIGHTGBM_PARAMS, "scale_pos_weight": SCALE_POS_WEIGHT
}
lightgbm_balanced_results = run_baseline(
    LGBMClassifier(**LIGHTGBM_BALANCED_PARAMS),
    run_name="lightgbm_balanced",
    description="LightGBM avec pondération de la classe 1",
    model_params=LIGHTGBM_BALANCED_PARAMS,
    weighting="balanced",
)

,Prédit : bon (0),Prédit : mauvais (1)
Réel : bon (0),56399,138
Réel : mauvais (1),4782,183


TN : 56399 | FP : 138 | FN : 4782 | TP : 183


,Prédit : bon (0),Prédit : mauvais (1)
Réel : bon (0),40741,15796
Réel : mauvais (1),1535,3430


TN : 40741 | FP : 15796 | FN : 1535 | TP : 3430
